In [ ]:
# Import

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings

warnings.filterwarnings('ignore')

# Matplotlib magyar karakterek támogatása
plt.rcParams['font.family'] = 'DejaVu Sans'
sns.set_style("whitegrid")
sns.set_palette("husl")

In [ ]:
# ADATOK BETÖLTÉSE ÉS ALAPVETŐ TISZTÍTÁS

# CSV betöltése (UTF-8 encoding a magyar karakterek miatt)
df = pd.read_csv('feedbacks.csv', sep=',')

print("=" * 80)
print("EREDETI ADATOK ÁTTEKINTÉSE")
print("=" * 80)
print(f"\nSorok száma: {len(df)}")
print(f"Oszlopok száma: {len(df.columns)}")
print("\nOszlopnevek:")
for i, col in enumerate(df.columns, 1):
    print(f"{i}. {col}")

# Oszlopnevek lerövidítése és tisztítása
column_mapping = {
    'Időbélyeg': 'timestamp',
    'Melyik országban élsz?': 'country',
    'Kb. hány éve fogadtál először sporteseményre?': 'years_betting',
    'Milyen gyakran fogadsz sporteseményekre?': 'frequency',
    'Miért szoktál fogadni leginkább? (több válasz is jelölhető)': 'motivation',
    'Hogyan szoktál tippeket szerezni most?': 'tip_source',
    'Mennyire megbízható a jelenlegi forrásod?': 'source_reliability',
    'Mi okozza számodra a legtöbb csalódást a sportfogadásban? (több válasz is jelölhető)': 'frustrations',
    'Mit gondolsz arról, hogy egy statisztikai algoritmus adja a tippeket?': 'algo_opinion',
    'Mit gondolsz az első havi profitgaranciáról (ha az első hónapban nem nyereséges a stratégia, visszakapod a díjat)?': 'guarantee_opinion',
    'Fontos számodra, hogy egy közösség része legyél, vagy inkább egyedül fogadsz?': 'community_preference',
    'Reálisan mennyi profitot várnál egy ilyen szolgáltatástól havonta? (lehetőleg egységben vagy százalékban megadva)': 'expected_profit',
    'Mennyi időt szeretnél rászánni a tippek követésére naponta?': 'time_commitment',
    'Mi tartana vissza attól, hogy előfizess? (több válasz is jelölhető)': 'barriers',
    'Egyes tippszolgáltatók Magyarországon 10 000–20 000 Ft-ot is elkérnek havonta. \nTe mennyire tartanád reálisnak, ha a mi szolgáltatásunk ennyibe kerülne?': 'price_perception',
    'Te pontosan mennyit tartanál reális havidíjnak?': 'willingness_to_pay',
    'Mennyire tartod reálisnak, hogy hosszú távon használnál egy ilyen szolgáltatást?': 'long_term_use',
    'Van bármi más gondolatod, amit fontosnak tartasz elmondani? (opcionális)': 'comments'
}

df.rename(columns={col_old: col_old.strip() for col_old in df.columns}, inplace=True)
df.rename(columns=column_mapping, inplace=True)

In [ ]:
# ADATTISZTÍTÁS ÉS TRANSZFORMÁCIÓ

print("\n" + "=" * 80)
print("ADATTISZTÍTÁS")
print("=" * 80)

# Timestamp konvertálás
df['timestamp'] = pd.to_datetime(df['timestamp'], format='%Y.%m.%d. %H:%M:%S')

# Országnevek standardizálása
df['country'] = df['country'].fillna('Ismeretlen')
df['country'] = df['country'].replace({
    'Magyar': 'Magyarország',
    'Magyarország ': 'Magyarország',
    'Magyarországon ': 'Magyarország'
})

# Évek tisztítása (numerikus értékké alakítás)
df['years_betting'] = pd.to_numeric(df['years_betting'], errors='coerce')

# Source reliability (1-5 skála)
df['source_reliability'] = pd.to_numeric(df['source_reliability'], errors='coerce')

# Long term use (1-5 skála)
df['long_term_use'] = pd.to_numeric(df['long_term_use'], errors='coerce')

# Willingness to pay tisztítása
def clean_wtp(value):
    if pd.isna(value):
        return np.nan
    value = str(value).strip()
    # Eltávolítjuk a szóközöket, pontokat
    value = value.replace(' ', '').replace('.', '').replace(',', '')
    # Számmá alakítás
    try:
        return float(value)
    except:
        return np.nan

df['willingness_to_pay'] = df['willingness_to_pay'].apply(clean_wtp)

# Expected profit tisztítása és kategorizálása
def parse_expected_profit(value):
    if pd.isna(value):
        return np.nan, np.nan
    
    value = str(value).lower().strip()
    
    # Százalékos értékek
    if '%' in value or 'százalék' in value:
        try:
            num = float(''.join(filter(str.isdigit, value)))
            return num, 'percentage'
        except:
            return np.nan, 'percentage'
    
    # Egység értékek
    if 'egység' in value:
        try:
            num = float(''.join(filter(str.isdigit, value)))
            return num, 'units'
        except:
            return np.nan, 'units'
    
    # Forint értékek (feltételezzük hogy ez is egység)
    if 'ft' in value or value.isdigit():
        try:
            num = float(''.join(filter(str.isdigit, value)))
            return num, 'huf'
        except:
            return np.nan, 'huf'
    
    # "Sokat" típusú válaszok
    if 'sok' in value:
        return np.nan, 'qualitative'
    
    return np.nan, np.nan

df[['expected_profit_value', 'expected_profit_type']] = df['expected_profit'].apply(
    lambda x: pd.Series(parse_expected_profit(x))
)

# Hiányzó értékek kezelése
print(f"\nHiányzó értékek oszloponként:")
missing = df.isnull().sum()
missing = missing[missing > 0]
if len(missing) > 0:
    print(missing)
else:
    print("Nincs hiányzó érték!")

In [ ]:
# ALAPSTATISZTIKÁK

print("\n" + "=" * 80)
print("ALAPSTATISZTIKÁK")
print("=" * 80)

print(f"\nVálaszadók száma: {len(df)}")
print(f"\nOrszág szerinti megoszlás:")
print(df['country'].value_counts())

print(f"\n\nFogadási gyakoriság:")
print(df['frequency'].value_counts())

print(f"\n\nFogadási tapasztalat (évek):")
print(df['years_betting'].describe())

print(f"\n\nFizetési hajlandóság (Ft/hó):")
wtp_stats = df['willingness_to_pay'].describe()
print(wtp_stats)
print(f"Medián: {df['willingness_to_pay'].median():.0f} Ft")

print(f"\n\nForrásmegbízhatóság (1-5):")
print(df['source_reliability'].describe())

print(f"\n\nHosszú távú használat hajlandósága (1-5):")
print(df['long_term_use'].describe())

In [ ]:
# KATEGORIZÁLÁS ÉS DUMMY VÁLTOZÓK

# Gyakoriság kategorizálása
frequency_order = {
    'Ritkán / csak nagyobb eseményekre': 1,
    'Havonta néhányszor': 2,
    'Hetente többször': 3,
    'Naponta': 4
}
df['frequency_score'] = df['frequency'].map(frequency_order)

# Aktivitási szintek
df['activity_level'] = pd.cut(
    df['frequency_score'],
    bins=[0, 1.5, 2.5, 4],
    labels=['Alkalmi', 'Rendszeres', 'Aktív']
)

# Tapasztalat kategorizálása
df['experience_level'] = pd.cut(
    df['years_betting'],
    bins=[0, 3, 10, 50],
    labels=['Kezdő (0-3 év)', 'Tapasztalt (4-10 év)', 'Veterán (10+ év)']
)

# WTP kategorizálása
df['wtp_segment'] = pd.cut(
    df['willingness_to_pay'],
    bins=[0, 3000, 6000, 10000, 25000],
    labels=['Alacsony (0-3k)', 'Közepes (3-6k)', 'Magas (6-10k)', 'Prémium (10k+)']
)

print("\n" + "=" * 80)
print("SZEGMENTÁCIÓ")
print("=" * 80)

print("\nAktivitási szintek:")
print(df['activity_level'].value_counts())

print("\nTapasztalati szintek:")
print(df['experience_level'].value_counts())

print("\nFizetési hajlandóság szegmensek:")
print(df['wtp_segment'].value_counts())

# Adatok mentése tisztított formában
df.to_csv('survey_data_cleaned.csv', index=False, encoding='utf-8')
print("\n✓ Tisztított adatok mentve: survey_data_cleaned.csv")

print("\n" + "=" * 80)
print("ADATTISZTÍTÁS BEFEJEZVE!")
print("=" * 80)